# Cross-model generalization: does a ChatGPT-trained classifier detect Claude text?

The model in `results/checkpoints/length_controlled/final` was trained only on
HC3 (human vs. **ChatGPT**, reddit_eli5 Q&A domain). Here we test it on a
completely different source: the **Claude**-generated essay subset of the
Kaggle DAIGT V2 dataset (`darragh_claude_v6` / `darragh_claude_v7`, ~2000
essays by Claude), paired against the human student essays in the same
dataset (`persuade_corpus`) that those Claude essays were written to match.

**Important confound, by design (see project decision log)**: this swaps
*both* the generator model (ChatGPT -> Claude) *and* the domain (Reddit Q&A ->
persuasive student essays) at once. A drop in accuracy here could be due to
either shift, not model-shift alone — we report this honestly as a limitation
rather than claiming a clean model-generalization result.

**Prerequisites**:
1. The `length_controlled` checkpoint downloaded locally (see `04_interpretability.ipynb`).
2. A Kaggle account with API credentials configured, either via
   `~/.kaggle/kaggle.json` or by running `kagglehub.login()` in the cell below.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import DistilBertForSequenceClassification

from model import load_tokenizer

CHECKPOINT_DIR = Path.cwd().parent / 'results' / 'checkpoints' / 'length_controlled' / 'final'
assert CHECKPOINT_DIR.exists(), f'{CHECKPOINT_DIR} not found — download it from Colab/Drive first (see 03_finetune_distilbert.ipynb).'

tokenizer = load_tokenizer(str(CHECKPOINT_DIR))
model = DistilBertForSequenceClassification.from_pretrained(str(CHECKPOINT_DIR))
model.eval()

## Download DAIGT V2 and pull out the Claude / human-essay subset

`thedrcat/daigt-v2-train-dataset` (44,868 rows total) contains a `source`
column identifying the generator: `persuade_corpus` for the original human
student essays, `darragh_claude_v6` / `darragh_claude_v7` for ~1000 Claude
essays each written against the same PERSUADE prompts.

In [ ]:
import kagglehub

# uncomment if kagglehub can't find credentials automatically:
# kagglehub.login()

daigt_path = kagglehub.dataset_download('thedrcat/daigt-v2-train-dataset')
csv_path = Path(daigt_path) / 'train_v2_drcat_02.csv'
daigt_df = pd.read_csv(csv_path)
print(daigt_df.shape)
daigt_df['source'].value_counts()

In [ ]:
CLAUDE_SOURCES = ['darragh_claude_v6', 'darragh_claude_v7']
HUMAN_SOURCE = 'persuade_corpus'

claude_df = daigt_df[daigt_df['source'].isin(CLAUDE_SOURCES)][['text', 'label']].copy()
human_df = daigt_df[daigt_df['source'] == HUMAN_SOURCE][['text', 'label']].copy()

# balance classes: subsample human essays down to the Claude subset size
human_df = human_df.sample(n=len(claude_df), random_state=42)

cross_df = pd.concat([claude_df, human_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print('claude (label should be 1):', claude_df['label'].unique())
print('human (label should be 0):', human_df['label'].unique())
print('cross_df shape:', cross_df.shape, 'balance:', cross_df['label'].value_counts().to_dict())

## Evaluate the ChatGPT-trained model on the Claude/human-essay set

In [ ]:
@torch.no_grad()
def predict_labels(model, tokenizer, texts, max_length=256, batch_size=16):
    preds = []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i:i + batch_size])
        enc = tokenizer(batch, truncation=True, padding=True, max_length=max_length, return_tensors='pt')
        logits = model(**enc).logits
        preds.extend(torch.argmax(logits, dim=-1).tolist())
    return preds


cross_df['pred'] = predict_labels(model, tokenizer, cross_df['text'])

precision, recall, f1, _ = precision_recall_fscore_support(cross_df['label'], cross_df['pred'], average='binary')
acc = accuracy_score(cross_df['label'], cross_df['pred'])
cross_metrics = {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}
print('Cross-model (Claude essays) metrics:', cross_metrics)

## Compare against in-domain (HC3/ChatGPT) test performance

Loads the same model's held-out HC3 test set (saved during training) to show
the in-domain vs. cross-model/cross-domain gap side by side.

In [ ]:
hc3_test_df = pd.read_parquet(CHECKPOINT_DIR / 'test_split.parquet')
hc3_test_df['pred'] = predict_labels(model, tokenizer, hc3_test_df['text'])

precision, recall, f1, _ = precision_recall_fscore_support(hc3_test_df['label'], hc3_test_df['pred'], average='binary')
acc = accuracy_score(hc3_test_df['label'], hc3_test_df['pred'])
in_domain_metrics = {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

comparison = pd.DataFrame([
    {'setting': 'in-domain (HC3, ChatGPT, reddit_eli5)', **in_domain_metrics},
    {'setting': 'cross-model+domain (DAIGT, Claude, essays)', **cross_metrics},
])
comparison

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(cross_df['label'], cross_df['pred'])
print('confusion matrix (rows=true, cols=pred), [0=human, 1=ai]:')
print(cm)
print(f"\nhuman-essay recall (specificity): {cm[0, 0] / cm[0].sum():.3f}")
print(f"claude-essay recall (sensitivity): {cm[1, 1] / cm[1].sum():.3f}")

import json
results_dir = Path.cwd().parent / 'results'
with open(results_dir / 'metrics.json', 'a', encoding='utf-8') as f:
    f.write(json.dumps({'cross_model_test': comparison.to_dict(orient='records')}) + '\n')

## Interpretation notes for the write-up

- Any accuracy drop here reflects **domain shift (Q&A -> essays) and model
  shift (ChatGPT -> Claude) combined** — this setup cannot isolate which one
  drives it. Say so explicitly rather than calling this a clean
  "cross-model" result.
- If specificity (human-essay recall) holds but sensitivity (Claude recall)
  drops, that's consistent with the model's learned "AI style" being
  ChatGPT-specific rather than a general AI-text detector.
- Worth cross-referencing with `04_interpretability.ipynb`: if the
  length_controlled model's top attributed tokens are HC3/Reddit-register
  words (contractions, informal phrasing) rather than general AI-generation
  markers, that's independent evidence the model learned something
  domain-specific, not just "AI-ness."